# ABCDFSS Evaluation on Google Colab / Kaggle with GPU
## Hướng dẫn chạy thử nghiệm Phân đoạn Few-Shot Miền chéo (CD-FSS) với Dữ liệu Google Drive

Notebook này hỗ trợ chạy thực nghiệm cho 2 dataset **Deepglobe** và **FSS-1000** với GPU (T4 / V100 / A100).

### Bước 1: Kiểm tra GPU & Môi trường

In [ ]:
!nvidia-smi
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))

### Bước 2: Clone Code từ GitHub hoặc Thiết lập Thư mục

In [ ]:
# Thay đường dẫn repo GitHub của bạn vào đây nếu chạy trên Google Colab mới:
# !git clone https://github.com/<YOUR_USERNAME>/ABCDFSS.git
# %cd ABCDFSS

!pip install opencv-python-headless tensorboardX gdown

### Bước 3: Kết nối Dữ liệu từ Google Drive

Bạn có thể chọn **Cách 1 (Mount Google Drive)** hoặc **Cách 2 (Tải file zip bằng link Google Drive)**:

#### Cách 1: Mount trực tiếp Google Drive (Khuyên dùng trên Google Colab)

In [ ]:
# Mount Drive vào Colab
from google.colab import drive
drive.mount('/content/drive')

# Đường dẫn tới thư mục dữ liệu trên Drive của bạn
# Thay đổi đường dẫn bên dưới theo vị trí thực tế trên Drive của bạn:
DRIVE_DATASET_DIR = '/content/drive/MyDrive/datasets'

# Tạo thư mục cục bộ và tạo symbolic link hoặc copy để tốc độ đọc ảnh nhanh nhất:
!mkdir -p ./datasets
# Ví dụ tạo symlink:
# !ln -s "$DRIVE_DATASET_DIR/deepglobe" ./datasets/deepglobe
# !ln -s "$DRIVE_DATASET_DIR/fss1000" ./datasets/fss1000

#### Cách 2: Tải file nén .zip từ link Google Drive bằng gdown (Chạy được cả Colab & Kaggle)

In [ ]:
import gdown
import os
import shutil

os.makedirs('./datasets', exist_ok=True)

# 1. Điền File ID của Deepglobe trên Google Drive
# (Lấy từ link: https://drive.google.com/file/d/<FILE_ID>/view?usp=sharing)
DEEPGLOBE_FILE_ID = 'DIEN_FILE_ID_DEEPGLOBE_VAO_DAY'

if DEEPGLOBE_FILE_ID != 'DIEN_FILE_ID_DEEPGLOBE_VAO_DAY':
    print('Đang tải Deepglobe...')
    gdown.download(id=DEEPGLOBE_FILE_ID, output='./datasets/deepglobe.zip', quiet=False)
    !unzip -q ./datasets/deepglobe.zip -d ./datasets/deepglobe
    print('Deepglobe sẵn sàng!')

# 2. Điền File ID của FSS-1000 trên Google Drive
FSS1000_FILE_ID = 'DIEN_FILE_ID_FSS1000_VAO_DAY'

if FSS1000_FILE_ID != 'DIEN_FILE_ID_FSS1000_VAO_DAY':
    print('Đang tải FSS-1000...')
    gdown.download(id=FSS1000_FILE_ID, output='./datasets/fss1000.zip', quiet=False)
    !unzip -q ./datasets/fss1000.zip -d ./datasets/fss1000
    print('FSS-1000 sẵn sàng!')

### Bước 4: Kiểm tra cấu trúc thư mục dữ liệu

In [ ]:
!ls -la ./datasets
!ls -la ./datasets/deepglobe || true
!ls -la ./datasets/fss1000 || true

### Bước 5: Chạy Thử Nghiệm Trên GPU

#### 5.1. Chạy Deepglobe (1-shot) - Mô hình Cải tiến (Đề xuất)

In [ ]:
!python main.py \
    --benchmark deepglobe \
    --datapath ./datasets/deepglobe \
    --nshot 1 \
    --adapt-to first-episode \
    --adapter-type depthwise_separable_3x3 \
    --fusion-mode softmax_margin \
    --verbosity 1

#### 5.2. Chạy Deepglobe (1-shot) - Baseline Cũ của tác giả (So sánh đối chứng)

In [ ]:
!python main.py \
    --benchmark deepglobe \
    --datapath ./datasets/deepglobe \
    --nshot 1 \
    --adapt-to first-episode \
    --adapter-type conv1x1 \
    --fusion-mode mean \
    --verbosity 1

#### 5.3. Chạy FSS-1000 (1-shot) - Mô hình Cải tiến

In [ ]:
!python main.py \
    --benchmark fss \
    --datapath ./datasets/fss1000 \
    --nshot 1 \
    --adapt-to first-episode \
    --adapter-type depthwise_separable_3x3 \
    --fusion-mode softmax_margin \
    --verbosity 1

#### 5.4. Chạy Deepglobe (5-shot)

In [ ]:
!python main.py \
    --benchmark deepglobe \
    --datapath ./datasets/deepglobe \
    --nshot 5 \
    --adapt-to first-episode \
    --adapter-type depthwise_separable_3x3 \
    --fusion-mode softmax_margin \
    --verbosity 1